# Redes líquidas (LTC): laboratorio reproducible y aplicaciones
**Sebastian Laverde,   Claudio de la Rosa · Ejercicio de clase ·Deep Learning I**

Este cuaderno está dirigido a los estudiantes de la materia de Deep Learning en el Doctorado de IA. Cada bloque explica qué hacemos, por qué y cómo interpretar su salida.

**Tres niveles de evidencia:** A) experimentos con los scripts originales y datos originales; B) simulaciones explicativas y traducción parcial a Python del análisis MATLAB; C) aplicaciones propuestas con datos sintéticos.

**Importante:** el cuaderno contiene código para realizar las corridas completas; no contiene resultados completos de cinco entrenamientos de 200 épocas ya realizados. Nunca sustituimos mediciones por cifras inventadas.

## Ruta de clase
1. Entender secuencias, memoria y velocidad de respuesta.
2. Observar una neurona LTC y su estabilidad.
3. Preparar el entorno original y descargar los datos.
4. Reproducir experimentos, guardar salidas y comparar con la publicación.
5. Explorar trayectorias internas.
6. Adaptar a psicología, medición estadística, cáncer, neurología y drones.
7. Exportar resultados y redactar una interpretación prudente.

En Colab: **Archivo → Subir cuaderno**. Empezar con CPU. Ejecutar de arriba hacia abajo. Las celdas marcadas como opcionales requieren activar un interruptor. Guardar el cuaderno en Drive antes de entrenamientos largos.

## 1. Vocabulario
| Término | Significado en este laboratorio |
|---|---|
| Entrada o característica | Medición que recibe el modelo: temperatura, tiempo de reacción, área de una lesión. |
| Secuencia | Mediciones en orden: ensayo 1, 2, 3… o visitas en fechas sucesivas. |
| Estado interno | Resumen numérico que el modelo actualiza al recibir cada medición. |
| Neurona | Unidad de cálculo; no una célula biológica real. |
| Peso | Número ajustado durante el aprendizaje. |
| Época | Un recorrido por los ejemplos de entrenamiento. |
| Lote | Grupo de ejemplos procesados juntos. |
| Pérdida | Medida del error que intentamos reducir. |
| Validación | Datos para seleccionar una versión del modelo. |
| Prueba | Datos reservados para evaluar la versión seleccionada. |
| Semilla | Número que permite controlar algunas fuentes de aleatoriedad. |

Una red recurrente conserva un estado entre pasos. En una LTC cambia también la rapidez con que ese estado responde. **La rapidez adaptable durante la inferencia no implica que los pesos se reentrenen automáticamente.**

In [ ]:
# Las bibliotecas modernas sirven para tablas y gráficos; el entrenamiento original va aparte.
import os, sys, json, time, pathlib, subprocess, urllib.request, zipfile, tarfile, hashlib, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from IPython.display import display
BASE = pathlib.Path('/content/ltc_clase') if pathlib.Path('/content').exists() else pathlib.Path.cwd() / 'ltc_clase'
BASE.mkdir(parents=True, exist_ok=True)
OUT = BASE / 'resultados'; OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.figsize':(9,4), 'axes.grid':True})
print('Carpeta de trabajo:', BASE)

## 2. Un proceso neuronal: “rápido” y “lento”
La constante de tiempo se parece al tiempo que tarda un termómetro en ajustarse a un cambio. Un valor pequeño produce una respuesta rápida. La fórmula simplificada del artículo es:

$$\frac{dx}{dt}=-(1/\tau+f)x+fA,\qquad \tau_{efectiva}=\frac{\tau}{1+\tau f}.$$

`x` es el estado; `τ` una escala temporal base; `f` una compuerta que responde a la entrada y al estado; `A` un valor de referencia. La derivada solo dice **cuánto cambia el estado**. En la siguiente demostración usamos una compuerta positiva simplificada; no es una red entrenada ni toda la implementación del artículo.

El paso semiimplícito calcula: `nuevo = (actual + dt*f*A) / (1 + dt*(1/tau + f))`.

In [ ]:
dt=0.01; t=np.arange(0,10,dt)
entrada=np.where((t>=2)&(t<6),1.0,0.0)
f=0.1+4*entrada  # Compuerta positiva: estas condiciones evitan amortiguación negativa.
tau=2.; A=1.; x=np.zeros_like(t); fijo=np.zeros_like(t)
for i in range(1,len(t)):
    x[i]=(x[i-1]+dt*f[i]*A)/(1+dt*(1/tau+f[i]))
    fijo[i]=(fijo[i-1]+dt*entrada[i]/tau)/(1+dt/tau)
tau_ef=tau/(1+tau*f)
fig,axs=plt.subplots(2,1,figsize=(9,6),sharex=True)
axs[0].plot(t,entrada,label='Entrada'); axs[0].plot(t,x,label='Estado líquido simplificado')
axs[0].plot(t,fijo,label='Filtro con constante fija'); axs[0].legend(); axs[0].set_ylabel('Valor')
axs[1].plot(t,tau_ef); axs[1].set_ylabel('Constante efectiva'); axs[1].set_xlabel('Tiempo')
plt.tight_layout(); plt.savefig(OUT/'neurona.png',dpi=140); plt.show()
assert np.isfinite(x).all() and x.min()>=0 and x.max()<=A

**Qué observar:** la constante efectiva baja cuando se activa la compuerta. Esto cambia la velocidad de respuesta. El gráfico es una ilustración de la ecuación; no demuestra que una red LTC supere a una LSTM.

### Estabilidad no equivale a exactitud
Un modelo puede permanecer acotado y predecir mal. Las garantías matemáticas del artículo dependen de las condiciones de los teoremas; no se extienden automáticamente a cualquier activación, peso, inicialización o discretización. A continuación probamos el paso simplificado con entradas grandes y compuertas positivas.

In [ ]:
estabilidad=[]
for ganancia in [0.1,1,10,1000]:
    z=0.; valores=[]
    for _ in range(1000):
        z=(z+dt*ganancia*A)/(1+dt*(1/tau+ganancia)); valores.append(z)
    estabilidad.append({'compuerta':ganancia,'máximo':max(valores),'fin':z,'constante':tau/(1+tau*ganancia)})
display(pd.DataFrame(estabilidad))

## 3. Reproducción: qué está disponible y qué falta
Usamos el repositorio oficial, fijado al commit `4a8a4fa22a91c4067b6944a264819ff30a4aae9d`. Nota: Descargar hoy sin fijar la versión puede cambiar el experimento mañana.

| Experimento | Script | Salida principal |
|---|---|---|
| Gestos | gesture.py | Accuracy |
| Ocupación | occupancy.py | Accuracy |
| Actividad con smartphone | har.py | Accuracy |
| MNIST como secuencia de filas | smnist.py | Accuracy |
| Tráfico | traffic.py | Error cuadrático |
| Electricidad | power.py | Error cuadrático |
| Ozono | ozone.py | F1 |
| Actividad de personas, configuración del script | person.py | Accuracy |
| Dinámica Half-Cheetah | cheetah.py | Error cuadrático |

Las condiciones adicionales del artículo (por ejemplo, segunda configuración de actividad personal y variantes de control) **no quedan automáticamente reproducidas** por ejecutar estos nueve archivos. No presentamos esto como una réplica exhaustiva de cada tabla y figura.

Hay dos discrepancias que deben documentarse: el suplemento describe 32 pasos para HAR y ocupación, pero los constructores de los scripts usan **16**. Además, el valor de aprendizaje de LTC depende del script (ocupación: 0.005; HAR: 0.01). Conservamos los scripts en la ruta principal; un cambio a 32 sería una corrida adicional declarada.

In [ ]:
COMMIT='4a8a4fa22a91c4067b6944a264819ff30a4aae9d'
REPO=BASE/'original_ltc'
def ejecutar(cmd,cwd=None):
    # Una lista de argumentos evita que el shell interprete caracteres del comando.
    subprocess.run([str(x) for x in cmd],cwd=cwd,check=True)
if not (REPO/'.git').exists():
    ejecutar(['git','clone','https://github.com/raminmh/liquid_time_constant_networks.git',REPO])
ejecutar(['git','checkout','--detach',COMMIT],cwd=REPO)
EXP=REPO/'experiments_with_ltcs'
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==COMMIT
print('Código original fijado a:',COMMIT)

## 4. TensorFlow 1.14 dentro de Colab: Versión para Python 3.14 (actual)
El código original requiere APIs antiguas. En vez de migrar las ecuaciones y las celdas a otra biblioteca, creamos un **segundo intérprete Python 3.7**, que ejecuta los scripts originales como subprocesos. El cuaderno conserva el Python moderno para tablas y gráficos.

Esto necesita descargas y espacio, puede tardar varios minutos y depende de que los servidores mantengan los paquetes. Usamos CPU, como los scripts originales. No hace falta activar GPU. Si falla una instalación, detenerse y leer el mensaje; no mezclar paquetes antiguos en el Python principal.

La versión de Ubuntu y las bibliotecas de sistema de Colab siguen siendo diferentes de las documentadas por los autores. Por tanto, ni siquiera este enfoque promete identidad bit a bit.

In [ ]:
# Preparación del intérprete legado. Se instala en una carpeta propia.
MAMBA=BASE/'bin'/'micromamba'; LEGACY=BASE/'legacy'; PY37=LEGACY/'bin'/'python'
if not MAMBA.exists():
    paquete=BASE/'micromamba.tar.bz2'
    urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/latest',paquete)
    with tarfile.open(paquete) as ar:
        miembro=ar.getmember('bin/micromamba')
        ar.extract(miembro,path=BASE)
    MAMBA.chmod(0o755)
if not PY37.exists():
    ejecutar([MAMBA,'create','-y','-p',LEGACY,'-c','conda-forge','python=3.7','pip=23.1.2'])
paquetes=['tensorflow==1.14.0','numpy==1.16.4','pandas==0.25.3',
          'scipy==1.3.3','protobuf==3.20.3','h5py==2.10.0','gast==0.2.2']
ejecutar([PY37,'-m','pip','install',*paquetes])
ejecutar([PY37,'-c',"import tensorflow as tf, numpy as np; print('TF:',tf.__version__,'NumPy:',np.__version__); assert tf.__version__=='1.14.0'"])
freeze=subprocess.check_output([str(PY37),'-m','pip','freeze'],text=True)
(OUT/'requirements_legacy.txt').write_text(freeze)

## 5. Descargar datos conservando sus archivos originales
Para no depender de rutas UCI antiguas, utilizamos también los enlaces actuales a ZIP. No cambiamos columnas ni etiquetas. Calculamos huellas SHA-256: si dos grupos usan huellas diferentes, primero deben investigar los datos.

**Primera sesión:** ocupación, porque tiene solo cinco características. HAR se incluye para continuar; tiene 561 características y puede ser bastante más costoso. El interruptor de datos adicionales permite descargar los otros archivos sin hacerlo obligatoriamente en la primera clase.

In [ ]:
DESCARGAR_ADICIONALES=False
DATA=EXP/'data'; DATA.mkdir(exist_ok=True)
def descargar_zip(url,destino):
    destino.mkdir(parents=True,exist_ok=True)
    archivo=BASE/(destino.name+'.zip')
    if not archivo.exists(): urllib.request.urlretrieve(url,archivo)
    with zipfile.ZipFile(archivo) as z:
        for n in z.namelist():
            if not (destino/n).resolve().is_relative_to(destino.resolve()):
                raise ValueError('Ruta ZIP inesperada')
        z.extractall(destino)
    return archivo
fuentes={
 'occupancy':'https://archive.ics.uci.edu/static/public/357/occupancy+detection.zip',
 'har':'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip',
 'gesture':'https://archive.ics.uci.edu/static/public/302/gesture+phase+segmentation.zip',
 'traffic':'https://archive.ics.uci.edu/static/public/492/metro+interstate+traffic+volume.zip',
 'power':'https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip',
 'ozone':'https://archive.ics.uci.edu/static/public/172/ozone+level+detection.zip',
 'person':'https://archive.ics.uci.edu/static/public/196/localization+data+for+person+activity.zip'}
huellas={}
for nombre in (list(fuentes) if DESCARGAR_ADICIONALES else ['occupancy','har']):
    archivo=descargar_zip(fuentes[nombre],DATA/nombre)
    huellas[nombre]=hashlib.sha256(archivo.read_bytes()).hexdigest()
# UCI puede envolver algunos contenidos en otro ZIP: extraer archivos ZIP internos.
for nombre in huellas:
    for interno in list((DATA/nombre).rglob('*.zip')):
        with zipfile.ZipFile(interno) as z: z.extractall(interno.parent)
(OUT/'huellas_datos.json').write_text(json.dumps(huellas,indent=2))
assert (DATA/'occupancy'/'datatraining.txt').exists(), 'Revisar estructura del ZIP de ocupación'
assert (DATA/'har'/'UCI HAR Dataset'/'train'/'X_train.txt').exists(), 'Revisar estructura de HAR'
display(pd.DataFrame(huellas.items(),columns=['Datos','SHA256']))

## 6. Mirar los datos antes de entrenar
Ocupación: temperatura, humedad, luz, CO₂ y razón de humedad. La etiqueta vale 0 (vacío) o 1 (ocupado). El modelo recibe mediciones de sensores, no fotos de personas.

HAR: 30 voluntarios realizan seis actividades con un smartphone. Cada fila contiene 561 características calculadas a partir de una ventana de sensores, no 561 instantes temporales. La división original entrenamiento/prueba se hizo por voluntarios; preservamos esos archivos. Fuente: [UCI HAR](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones).

In [ ]:
ocupacion=pd.read_csv(DATA/'occupancy'/'datatraining.txt')
display(ocupacion.head()); display(ocupacion.describe())
fig,axs=plt.subplots(2,1,figsize=(10,6),sharex=True)
axs[0].plot(ocupacion['CO2'].iloc[:500]); axs[0].set_ylabel('CO₂')
axs[1].step(np.arange(500),ocupacion['Occupancy'].iloc[:500]); axs[1].set_ylabel('Ocupación'); axs[1].set_xlabel('Registro')
plt.tight_layout(); plt.show()
print('Frecuencia de etiquetas:',ocupacion.Occupancy.value_counts(normalize=True).to_dict())

### Normalizar y formar secuencias
Normalizar significa expresar cuánto se aparta una medición del promedio, en unidades de su variabilidad. Los promedios y desviaciones deben aprenderse del entrenamiento. Aplicarlos a prueba no equivale a recalcularlos con prueba.

Los scripts usan el orden **[tiempo, lote, características]**. El lote agrupa muchas secuencias. Los ejemplos de adaptación usarán otra convención explícita.

**Limitación del código original:** se crean ventanas solapadas y después se separan entrenamiento y validación al azar. Ventanas diferentes pueden compartir registros. Esto afecta la independencia de la validación; la ruta de reproducción lo conserva. Para una aplicación nueva, dividir primero por persona/parcela/sesión y luego formar ventanas. Además, concatenar filas HAR puede producir secuencias que cruzan cambios de sujeto: revisar los identificadores antes de atribuir una interpretación fisiológica.

In [ ]:
# Ejemplo pequeño para ver el solapamiento: dos ventanas comparten tres registros.
ejemplo=np.arange(10)
ventanas=np.array([ejemplo[s:s+4] for s in range(len(ejemplo)-4)])
display(pd.DataFrame(ventanas,columns=['paso 1','paso 2','paso 3','paso 4']))
media=ocupacion[['Temperature','Humidity','Light','CO2','HumidityRatio']].mean()
desv=ocupacion[['Temperature','Humidity','Light','CO2','HumidityRatio']].std(ddof=0)
normal=(ocupacion[media.index]-media)/desv
display(normal.head())

## 7. Ejecutar los originales y conservar la trazabilidad
Se añade un lanzador externo que fija semillas y llama al script sin editar sus ecuaciones, pérdida ni optimizador. Cada ejecución tiene carpeta propia para que no se sobrescriban checkpoints o se mezclen filas CSV. Los archivos originales permanecen intactos.

**Modo corto:** 3 épocas, una semilla, tres modelos; verifica funcionamiento, no replica las cifras publicadas. **Modo completo:** 200 épocas, cinco semillas, los cinco modelos de comparación. El entrenamiento completo puede superar una sesión de Colab; descargar los resultados o copiar la carpeta a Drive entre sesiones. Las semillas elegidas aquí no son semillas originales documentadas del artículo.

El checkpoint se selecciona por validación. Los scripts imprimen prueba durante el entrenamiento; no usar esos valores para elegir hiperparámetros. Algunos scripts evalúan antes de la época y no evalúan después de la última: conservamos ese comportamiento. Con una sola época sus CSV pueden conservar ceros: usar al menos tres.

In [ ]:
# Lanzador escrito para Python 3.7; el kernel de Colab no importa TensorFlow antiguo.
launcher = r"""
import os,sys,runpy,random
os.environ['CUDA_VISIBLE_DEVICES']='-1'
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
os.environ['OMP_NUM_THREADS']='2'
os.environ['TF_NUM_INTRAOP_THREADS']='2'
os.environ['TF_NUM_INTEROP_THREADS']='2'
import numpy as np
import tensorflow as tf
script,seed=sys.argv[1],int(sys.argv[2])
random.seed(seed); np.random.seed(seed); tf.set_random_seed(seed)
sys.path.insert(0,os.path.dirname(script))
sys.argv=[script]+sys.argv[3:]
runpy.run_path(script,run_name='__main__')
"""
LANZADOR=BASE/'lanzador.py'; LANZADOR.write_text(launcher)
MODO_COMPLETO=False
DATASETS=['occupancy']  # Ampliar: har,gesture,traffic,power,ozone,smnist,person,cheetah
MODELOS=['ltc','lstm','ctrnn','node','ctgru'] if MODO_COMPLETO else ['ltc','lstm','ctrnn']
SEMILLAS=[11,22,33,44,55] if MODO_COMPLETO else [11]
EPOCAS=200 if MODO_COMPLETO else 3
print('Configuración:',DATASETS,MODELOS,SEMILLAS,'épocas:',EPOCAS)

In [ ]:
# Entrenamiento con salida guardada y avances resumidos por época.
# Reanudar esta celda omite corridas terminadas; cambiar configuración crea otra carpeta.
registros=[]
for dataset in DATASETS:
    for modelo in MODELOS:
        for semilla in SEMILLAS:
            carpeta=OUT/'corridas'/f'{dataset}_{modelo}_s{semilla}_e{EPOCAS}'
            carpeta.mkdir(parents=True,exist_ok=True)
            enlace=carpeta/'data'
            if not enlace.exists(): enlace.symlink_to(DATA.resolve(),target_is_directory=True)
            meta={'dataset':dataset,'modelo':modelo,'semilla':semilla,'epocas':EPOCAS,
                  'unidades':32,'commit':COMMIT,'modo_completo':MODO_COMPLETO}
            terminado=carpeta/'terminado.json'
            if not terminado.exists():
                cmd=[str(PY37),'-u',str(LANZADOR),str(EXP/(dataset+'.py')),str(semilla),
                     '--model',modelo,'--epochs',str(EPOCAS),'--size','32','--log','1']
                inicio=time.time(); print('Iniciando',carpeta.name,flush=True)
                env=os.environ.copy(); env['PYTHONHASHSEED']=str(semilla)
                with (carpeta/'log.txt').open('w') as log:
                    proceso=subprocess.Popen(cmd,cwd=carpeta,stdout=subprocess.PIPE,
                                             stderr=subprocess.STDOUT,text=True,env=env)
                    for linea in proceso.stdout:
                        log.write(linea); log.flush()
                        if 'Epochs ' in linea or 'Best epoch ' in linea: print(linea.strip(),flush=True)
                    retorno=proceso.wait()
                if retorno:
                    raise RuntimeError(f'Falló {carpeta.name}. Revisar {carpeta}/log.txt')
                meta['segundos']=time.time()-inicio
                terminado.write_text(json.dumps(meta,indent=2))
            registros.append(json.loads(terminado.read_text()))
pd.DataFrame(registros).to_csv(OUT/'ejecuciones.csv',index=False)

## 8. Comparar mediciones con referencias publicadas
Accuracy es proporción de aciertos; F1 combina precisión y sensibilidad; error cuadrático penaliza más los errores grandes. No comparar numéricamente métricas diferentes. La desviación estándar describe variación entre corridas, no incertidumbre clínica ni variabilidad entre pacientes.

El artículo reporta cinco corridas por condición. **HAR es un contraejemplo útil: LTC no es el ganador allí.** Más expresividad no implica ganar en toda tarea. Las cifras siguientes son referencias transcritas de la Tabla 3, no resultados producidos por este cuaderno.

In [ ]:
# Referencias: media y desviación estándar de la Tabla 3. Accuracy en porcentaje; F1 en 0–1.
referencias={
 'gesture':{'lstm':(64.57,.59),'ctrnn':(59.01,1.22),'node':(46.97,3.03),'ctgru':(68.31,1.78),'ltc':(69.55,1.13)},
 'occupancy':{'lstm':(93.18,1.66),'ctrnn':(94.54,.54),'node':(90.15,1.71),'ctgru':(91.44,1.67),'ltc':(94.63,.17)},
 'har':{'lstm':(95.85,.29),'ctrnn':(95.73,.47),'node':(97.26,.10),'ctgru':(96.16,.39),'ltc':(95.67,.575)},
 'smnist':{'lstm':(98.41,.12),'ctrnn':(96.73,.19),'node':(97.61,.14),'ctgru':(98.27,.14),'ltc':(97.57,.18)},
 'traffic':{'lstm':(.169,.004),'ctrnn':(.224,.008),'node':(1.512,.179),'ctgru':(.389,.076),'ltc':(.099,.0095)},
 'power':{'lstm':(.628,.003),'ctrnn':(.742,.005),'node':(1.254,.149),'ctgru':(.586,.003),'ltc':(.642,.021)},
 'ozone':{'lstm':(.284,.025),'ctrnn':(.236,.011),'node':(.168,.006),'ctgru':(.260,.024),'ltc':(.302,.0155)}}
filas=[]
for dataset,modelos in referencias.items():
    for modelo,(media,sd) in modelos.items(): filas.append({'dataset':dataset,'modelo':modelo,'publicado':media,'sd_publicada':sd})
REF=pd.DataFrame(filas); display(REF); REF.to_csv(OUT/'referencias_tabla3.csv',index=False)
# Los scripts pueden tener nombres distintos para métricas: mostrarlos antes de seleccionar.
mediciones=[]
for carpeta in sorted((OUT/'corridas').glob('*')):
    if not (carpeta/'terminado.json').exists(): continue
    meta=json.loads((carpeta/'terminado.json').read_text())
    for csv in (carpeta/'results').rglob('*.csv'):
        datos=pd.read_csv(csv); datos.columns=datos.columns.str.strip()
        if len(datos):
            fila=datos.iloc[-1].to_dict(); fila.update(meta); mediciones.append(fila)
MED=pd.DataFrame(mediciones)
display(MED)
if len(MED): MED.to_csv(OUT/'mediciones.csv',index=False)

In [ ]:
# Comparación automática para clasificación. No confundir F1, MSE y porcentaje.
if len(MED) and 'test accuracy' in MED.columns:
    resumen=MED.groupby(['dataset','modelo'])['test accuracy'].agg(['mean','std','count']).reset_index()
    comparacion=resumen.merge(REF,on=['dataset','modelo'],how='left')
    comparacion['diferencia_pp']=comparacion['mean']-comparacion['publicado']
    display(comparacion); comparacion.to_csv(OUT/'comparacion_accuracy.csv',index=False)
    for dataset,g in comparacion.groupby('dataset'):
        if dataset not in ['occupancy','har','gesture','smnist']: continue
        pos=np.arange(len(g)); plt.figure()
        plt.bar(pos-.2,g['mean'],width=.4,label='Medido en este cuaderno')
        plt.bar(pos+.2,g['publicado'],width=.4,label='Publicado, Tabla 3')
        plt.xticks(pos,g['modelo']); plt.ylabel('Accuracy (%)'); plt.title(dataset+' · '+('200 épocas' if MODO_COMPLETO else 'Prueba corta, no réplica numérica'))
        plt.legend(); plt.tight_layout(); plt.show()
else:
    print('No hay accuracy; revisar las columnas MSE/F1 de los CSV para estas tareas.')

### Cómo explicar un resultado que difiere
Antes de atribuir la diferencia al modelo, revisar: datos descargados, commit, longitud de secuencia 16/32, número de épocas, semillas, versión de bibliotecas y criterio de checkpoint. No ajustar parámetros mirando prueba para acercarnos a una cifra publicada.

La media de cinco semillas tampoco es una prueba estadística entre pacientes. Para comparar generalización en nuevas personas, hacen falta particiones por persona y evaluación de unidades independientes. En datos temporales, el bootstrap debe conservar grupos/bloques; remuestrear todos los pasos como independientes puede subestimar la incertidumbre.

### Métricas de los demás experimentos
Los nombres de columnas originales pueden ser confusos: en ozono, `test acc` contiene F1, no accuracy; en regresión, `test loss` es MSE y `test mae` es otra métrica. Usamos la pérdida MSE para compararla con las tablas de regresión. No renombramos silenciosamente archivos originales.

In [ ]:
mapa_metricas={
 'gesture':('test accuracy','Accuracy (%)',True), 'occupancy':('test accuracy','Accuracy (%)',True),
 'har':('test accuracy','Accuracy (%)',True), 'smnist':('test accuracy','Accuracy (%)',True),
 'person':('test accuracy','Accuracy (%)',True), 'traffic':('test loss','MSE',False),
 'power':('test loss','MSE',False), 'cheetah':('test loss','MSE',False), 'ozone':('test acc','F1',True)}
filas_metricas=[]
if len(MED):
    for dataset,g in MED.groupby('dataset'):
        columna,nombre,mayor=mapa_metricas[dataset]
        if columna not in g: continue
        for modelo,h in g.groupby('modelo'):
            valores=pd.to_numeric(h[columna],errors='coerce').dropna()
            if not len(valores): continue
            ref=referencias.get(dataset,{}).get(modelo,(np.nan,np.nan))
            filas_metricas.append({'dataset':dataset,'modelo':modelo,'metrica':nombre,
                'media_medida':valores.mean(),'sd_medida':valores.std(ddof=1) if len(valores)>1 else np.nan,
                'n':len(valores),'publicado_tabla3':ref[0],'mejor_si':'sube' if mayor else 'baja'})
    TODAS=pd.DataFrame(filas_metricas); display(TODAS)
    TODAS.to_csv(OUT/'comparacion_todas_metricas.csv',index=False)

### Curvas de aprendizaje
Una pérdida que baja en entrenamiento mientras sube en validación sugiere sobreajuste. Las curvas de prueba se mantienen en los logs originales para trazabilidad; no las usamos para decidir parámetros. Los valores de validación de estos logs corresponden a la evaluación al comienzo de la época.

In [ ]:
import re
patron=r'Epochs (\d+), train loss: ([\d.eE+-]+), train accuracy: ([\d.eE+-]+)%, valid loss: ([\d.eE+-]+), valid accuracy: ([\d.eE+-]+)%'
for carpeta in sorted((OUT/'corridas').glob('*')):
    if not (carpeta/'log.txt').exists(): continue
    coincidencias=re.findall(patron,(carpeta/'log.txt').read_text())
    if not coincidencias: continue
    curva=pd.DataFrame(coincidencias,columns=['epoca','train_loss','train_accuracy','valid_loss','valid_accuracy']).astype(float)
    curva.to_csv(carpeta/'curva.csv',index=False)
    fig,axs=plt.subplots(1,2,figsize=(10,3))
    curva.plot(x='epoca',y=['train_loss','valid_loss'],ax=axs[0])
    curva.plot(x='epoca',y=['train_accuracy','valid_accuracy'],ax=axs[1])
    axs[0].set_ylabel('Pérdida'); axs[1].set_ylabel('Accuracy (%)')
    fig.suptitle(carpeta.name); plt.tight_layout(); plt.show()

## 9. Trayectorias internas: traducción parcial del MATLAB original
Esta sección traduce la configuración **de una capa** de `main.m`, `node_def.m`, `ctrnn_def.m` y `ltc_def.m` a NumPy/SciPy. No entrena pesos; compara sistemas con pesos aleatorios compartidos y entrada circular. Usamos hard-tanh, 25 unidades, desviación de pesos 2, de sesgos 1, τ positivos y tolerancias 1e-2/1e-4 como el archivo MATLAB.

En esos archivos, NODE y CT-RNN calculan una activación recurrente pero **no la usan** en la derivada; LTC sí la usa. Conservamos ese detalle. La distribución aleatoria de NumPy y RK45 de SciPy no producen los mismos números que MATLAB. Se puede estudiar el procedimiento; no afirmamos réplica exacta de las Figuras 2–3.

PCA resume estados de muchas unidades en dos ejes. La longitud del recorrido es una medida operativa de expresividad. No mide inteligencia, conciencia, validez psicológica ni precisión diagnóstica.

In [ ]:
def trayectorias(seed=11,k=25,sigma=2):
    rng=np.random.RandomState(seed); gt=np.linspace(0,2*np.pi,100)
    entradas=np.array([np.sin(gt),np.cos(gt)])
    W=rng.normal(0,sigma,(k,k)); b=rng.normal(0,1,k)
    E=rng.normal(0,sigma,(k,k)); tau=np.maximum(np.abs(rng.normal(0,1,k)),1e-8)
    Wr=rng.normal(0,sigma,(k,k)); br=rng.normal(0,1,k); Er=rng.normal(0,sigma,(k,k))
    def rhs(tipo):
        def funcion(t,y):
            u=np.array([np.interp(t,gt,canal) for canal in entradas])
            ff=np.clip(W[:2,:].T@u+b,-1,1)
            rec=np.clip(Wr.T@y+br,-1,1)
            if tipo=='node': return ff
            if tipo=='ctrnn': return -y/tau+ff
            return -y*(1/tau+np.abs(ff)+np.abs(rec))+E.sum(axis=0)*ff+Er.sum(axis=0)*rec
        return funcion
    resultados={}
    for tipo in ['node','ctrnn','ltc']:
        # solve_ivp sin t_eval conserva nodos adaptativos, como las salidas de ode45.
        sol=solve_ivp(rhs(tipo),(0,2*np.pi),np.zeros(k),rtol=1e-2,atol=1e-4,dense_output=True)
        if not sol.success: raise RuntimeError(sol.message)
        proyeccion=PCA(n_components=2).fit(sol.y.T)
        z=proyeccion.transform(sol.y.T)
        longitud=np.linalg.norm(np.diff(z,axis=0),axis=1).sum()
        # Malla común adicional: informa sensibilidad de PCA/longitud al muestreo del solver.
        Y=sol.sol(np.linspace(0,2*np.pi,500)).T
        Z=PCA(n_components=2).fit_transform(Y)
        resultados[tipo]={'z':z,'longitud':longitud,'nodos':len(sol.t),
            'varianza_2pc':proyeccion.explained_variance_ratio_.sum(),
            'longitud_malla_comun':np.linalg.norm(np.diff(Z,axis=0),axis=1).sum()}
    return resultados
tr=trayectorias()
fig,axs=plt.subplots(1,3,figsize=(12,4))
for ax,(tipo,r) in zip(axs,tr.items()):
    ax.plot(r['z'][:,0],r['z'][:,1]); ax.set_title(f"{tipo}: longitud {r['longitud']:.2f}")
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout(); plt.savefig(OUT/'trayectorias.png',dpi=140); plt.show()
display(pd.DataFrame([{k:v for k,v in r.items() if k!='z'}|{'modelo':tipo} for tipo,r in tr.items()]))

In [ ]:
# Repetir con varias semillas: una figura aislada no establece un patrón general.
longitudes=[]
for seed in range(10):
    for tipo,r in trayectorias(seed).items():
        longitudes.append({'semilla':seed,'modelo':tipo,'longitud':r['longitud'],
                           'longitud_malla_comun':r['longitud_malla_comun']})
L=pd.DataFrame(longitudes); display(L.groupby('modelo').agg({'longitud':['mean','std'],'longitud_malla_comun':['mean','std']}))
L.boxplot(column='longitud',by='modelo'); plt.suptitle(''); plt.title('Traducción parcial · 10 semillas'); plt.ylabel('Longitud PCA'); plt.show()
L.to_csv(OUT/'trayectorias_semillas.csv',index=False)

## 10. ¿Cómo adaptar LTC a otras áreas?
LTC tiene una motivación clara cuando importan **orden, historia y evolución**. Una tabla de pacientes sin orden temporal no se vuelve una serie por ordenar sus filas. Una foto aislada tampoco constituye un seguimiento.

| Área | Unidad independiente | Entrada secuencial | Objetivo posible | Comparación básica |
|---|---|---|---|---|
| Pruebas psicológicas | Participante | Ensayos: respuesta, latencia, dificultad | Rendimiento siguiente, fatiga operativa | Media móvil, regresión, LSTM |
| Medición estadística | Instrumento o serie | Lecturas y condiciones ambientales | Señal futura o error de medición | Modelo lineal, filtro/Kalman |
| Cáncer en imágenes | Paciente | Visitas: características de lesión, tratamiento | Evolución de volumen | Último valor, radiomics + regresión |
| Neurología en imágenes | Paciente | Visitas MRI y otras medidas | Evolución o clasificación validada | Modelo clínico, CNN/3D CNN |
| Drones y arbustos | Parcela | Fechas de vuelo: cobertura, altura, conteos | Crecimiento o cambio de cobertura | Tendencia lineal, último valor |

Las filas de una imagen o los cortes de una MRI pueden tratarse como una secuencia **espacial**; no equivalen a tiempo. Es otra hipótesis, que debe compararse con modelos espaciales. Para imágenes longitudinales: imagen → extractor/segmentación → vector por visita → LTC → salida. LTC no reemplaza por sí sola la segmentación ni reconoce tumores automáticamente.

A continuación ejecutamos ejemplos sintéticos para enseñar la mecánica. No se usan imágenes clínicas reales ni datos de pacientes.

## 11. Imágenes didácticas: de píxeles a mediciones
Construimos fotografías esquemáticas de vegetación y mapas esquemáticos de lesiones. Un umbral identifica una región; contamos píxeles y extraemos proporción de área e intensidad. Es una simplificación deliberada. En imágenes reales se necesitan anotaciones, registro entre visitas, calibración espacial y validación del segmentador.

Para drones: área en m² = píxeles × GSD², si el tamaño de píxel en terreno es uniforme y el mapa está correctamente georreferenciado. **Cobertura no es número de arbustos:** copas que se superponen no se cuentan bien con componentes conectados. Sin correspondencia temporal, no se distingue crecimiento de una copa del desplazamiento del vuelo.

In [ ]:
from scipy.ndimage import label
rng_img=np.random.default_rng(7)
def imagen_sintetica(tipo,radio,n=64):
    yy,xx=np.mgrid[:n,:n]; disco=(xx-n/2)**2+(yy-n/2)**2<radio**2
    if tipo=='vegetacion':
        img=np.full((n,n,3),.20); img[:,:,0]=.35; img[disco]=[.10,.75,.12]
    else:
        img=np.full((n,n,3),.15); img[disco]=[.85,.85,.85]
    return np.clip(img+rng_img.normal(0,.02,img.shape),0,1)
def medir_imagen(img,tipo,gsd=None):
    if tipo=='vegetacion': mascara=(img[:,:,1]>img[:,:,0]+.15)&(img[:,:,1]>.4)
    else: mascara=img.mean(axis=2)>.5
    componentes,numero=label(mascara)
    area_px=int(mascara.sum()); cobertura=float(mascara.mean())
    intensidad=float(img[mascara].mean()) if area_px else 0.
    return {'area_px':area_px,'proporcion':cobertura,'intensidad':intensidad,
            'componentes':numero,'area_m2':area_px*gsd**2 if gsd is not None else np.nan}
fig,axs=plt.subplots(2,3,figsize=(10,6)); medidas=[]
for fila,tipo in enumerate(['lesion','vegetacion']):
    for j,radio in enumerate([6,10,15]):
        img=imagen_sintetica(tipo,radio); axs[fila,j].imshow(img); axs[fila,j].axis('off')
        m=medir_imagen(img,tipo,.02 if tipo=='vegetacion' else None)
        axs[fila,j].set_title(f"{tipo}: {m['area_px']} píxeles"); medidas.append({'tipo':tipo,'visita':j,**m})
plt.tight_layout(); plt.show(); display(pd.DataFrame(medidas))

## 12. Cinco ejemplos sintéticos de adaptación
Cada entidad tiene nueve observaciones equidistantes. Ocho son entradas; la novena es el objetivo. Separamos entidades completas: ninguna persona/parcela aparece en ambos conjuntos. Para psicología predecimos latencia siguiente; para medición una señal observada futura; para cáncer área esquemática; para neurología un marcador numérico simulado; para drones cobertura.

El ejemplo neurológico no detecta una enfermedad: ilustra cómo organizar un marcador longitudinal. No equivale a un biomarcador validado. El ejemplo de cáncer no establece malignidad. La medición estadística predictiva no demuestra validez del instrumento ni identifica una variable latente.

**Intervalos irregulares:** aquí son iguales. El script original asume pasos uniformes; agregar una columna de días no convierte automáticamente el solucionador en un modelo con dt variable. Para datos reales se puede remuestrear dentro de cada entidad sin usar visitas futuras, o implementar explícitamente dt variable y documentar que es una modificación.

In [ ]:
def crear_ejemplo(area,n_entidades=90,seed=2026):
    rng=np.random.default_rng(seed); X=[]; y=[]
    for entidad in range(n_entidades):
        tiempo=np.arange(9,dtype=float); base=rng.uniform(.2,.6); pendiente=rng.uniform(-.015,.04)
        ruido=rng.normal(0,.01,9)
        if area=='psicologia':
            latencia=600+150*base+15*pendiente*tiempo*100+rng.normal(0,15,9)
            dificultad=np.sin(tiempo/3)+rng.normal(0,.05,9)
            serie=np.column_stack([latencia,dificultad]); objetivo=latencia[-1]
        elif area=='medicion':
            señal=base+pendiente*tiempo+ruido
            temperatura=20+np.sin(tiempo/4)+rng.normal(0,.1,9)
            serie=np.column_stack([señal,temperatura]); objetivo=señal[-1]
        elif area in ['cancer','drones']:
            radios=np.clip(6+base*5+pendiente*tiempo*10,2,25)
            tipo='lesion' if area=='cancer' else 'vegetacion'
            m=[medir_imagen(imagen_sintetica(tipo,r),tipo) for r in radios]
            serie=np.array([[v['proporcion'],v['intensidad']] for v in m]); objetivo=serie[-1,0]
        else:
            marcador=base-pendiente*tiempo+ruido
            puntuacion=25+marcador*4+rng.normal(0,.2,9)
            serie=np.column_stack([marcador,puntuacion]); objetivo=marcador[-1]
        X.append(serie[:8]); y.append(objetivo)
    return np.asarray(X,np.float32),np.asarray(y,np.float32)
AREAS=['psicologia','medicion','cancer','neurologia','drones']
for area in AREAS:
    X,y=crear_ejemplo(area); carpeta=OUT/'adaptaciones'/area; carpeta.mkdir(parents=True,exist_ok=True)
    orden=np.random.default_rng(123).permutation(len(X)); a,b,c=orden[:54],orden[54:72],orden[72:]
    # Todo escalamiento se ajusta exclusivamente con entrenamiento.
    mx=X[a].mean(axis=(0,1)); sx=np.maximum(X[a].std(axis=(0,1)),1e-6)
    my=y[a].mean(); sy=max(y[a].std(),1e-6)
    np.savez(carpeta/'datos.npz',Xtrain=(X[a]-mx)/sx,ytrain=(y[a]-my)/sy,
             Xvalid=(X[b]-mx)/sx,yvalid=(y[b]-my)/sy,Xtest=(X[c]-mx)/sx,ytest=(y[c]-my)/sy,
             media_y=my,sd_y=sy,ultimo_test=X[c,-1,0],objetivo_original=y[c])
    print(area,'[entidades, tiempo, variables]:',X.shape,'→ objetivo:',y.shape)

### Entrenar una LTC original sobre datos nuevos
El siguiente entrenador usa la clase `LTCCell` original, pero agrega una cabeza de regresión al último estado. Es **una adaptación didáctica propia**, no el procedimiento de los experimentos originales. Cada conjunto de adaptación usa su propia red; no trasladamos un modelo entrenado en ocupación al diagnóstico médico.

Comparamos con dos referencias: último valor observado y regresión Ridge sobre la secuencia aplanada. Un modelo complejo debe justificar su uso frente a estas referencias. Seleccionamos el checkpoint con validación y evaluamos prueba una vez al final.

In [ ]:
entrenador_adaptacion="\nimport os,sys,json\nos.environ['CUDA_VISIBLE_DEVICES']='-1'\nos.environ['TF_CPP_MIN_LOG_LEVEL']='2'\nos.environ['OMP_NUM_THREADS']='2'\nos.environ['TF_NUM_INTRAOP_THREADS']='2'\nos.environ['TF_NUM_INTEROP_THREADS']='2'\nimport numpy as np\nimport tensorflow as tf\nsys.path.insert(0,sys.argv[1])\nfrom ltc_model import LTCCell\nruta=sys.argv[2]; np.random.seed(42); tf.set_random_seed(42)\nd=np.load(os.path.join(ruta,'datos.npz'))\nx=tf.placeholder(tf.float32,[None,8,2]); target=tf.placeholder(tf.float32,[None])\ncell=LTCCell(8)\nh,_=tf.nn.dynamic_rnn(cell,x,dtype=tf.float32,time_major=False)\npred=tf.layers.Dense(1)(h[:,-1,:])[:,0]\nloss=tf.reduce_mean(tf.square(pred-target))\ntrain=tf.train.AdamOptimizer(.005).minimize(loss)\nconstrain=cell.get_param_constrain_op(); saver=tf.train.Saver()\nhistory=[]; best=float('inf')\nconfig=tf.ConfigProto(intra_op_parallelism_threads=2,inter_op_parallelism_threads=2)\nwith tf.Session(config=config) as sess:\n    sess.run(tf.global_variables_initializer())\n    for epoch in range(40):\n        order=np.random.permutation(len(d['Xtrain'])); losses=[]\n        for start in range(0,len(order),16):\n            idx=order[start:start+16]\n            l,_=sess.run([loss,train],{x:d['Xtrain'][idx],target:d['ytrain'][idx]})\n            sess.run(constrain); losses.append(float(l))\n        vl=sess.run(loss,{x:d['Xvalid'],target:d['yvalid']})\n        history.append({'epoch':epoch,'train_mse':float(np.mean(losses)),'valid_mse':float(vl)})\n        if vl<best:\n            best=vl; saver.save(sess,os.path.join(ruta,'modelo'))\n    saver.restore(sess,os.path.join(ruta,'modelo'))\n    yp=sess.run(pred,{x:d['Xtest']})*d['sd_y']+d['media_y']\nnp.save(os.path.join(ruta,'predicciones_ltc.npy'),yp)\nwith open(os.path.join(ruta,'historia.json'),'w') as f: json.dump(history,f)\nprint('Adaptación terminada:',ruta)\n"
ADAPTADOR=BASE/'entrenar_adaptacion.py'
ADAPTADOR.write_text(entrenador_adaptacion)
for area in AREAS:
    ejecutar([PY37,ADAPTADOR,EXP,OUT/'adaptaciones'/area])

In [ ]:
metricas=[]
for area in AREAS:
    carpeta=OUT/'adaptaciones'/area; d=np.load(carpeta/'datos.npz')
    # Ridge seleccionada por validación, sin usar prueba para elegir regularización.
    candidatos=[]
    for alpha in [.1,1,10,100]:
        modelo=Ridge(alpha=alpha).fit(d['Xtrain'].reshape(len(d['Xtrain']),-1),d['ytrain'])
        err=mean_squared_error(d['yvalid'],modelo.predict(d['Xvalid'].reshape(len(d['Xvalid']),-1)))
        candidatos.append((err,modelo))
    ridge=min(candidatos,key=lambda par:par[0])[1]
    yp_ridge=ridge.predict(d['Xtest'].reshape(len(d['Xtest']),-1))*d['sd_y']+d['media_y']
    real=d['objetivo_original']; yp_ltc=np.load(carpeta/'predicciones_ltc.npy')
    for nombre,yp in [('Último valor',d['ultimo_test']),('Ridge',yp_ridge),('LTC',yp_ltc)]:
        metricas.append({'area':area,'modelo':nombre,'MAE':mean_absolute_error(real,yp),
                        'RMSE':np.sqrt(mean_squared_error(real,yp))})
    historia=pd.read_json(carpeta/'historia.json'); historia.plot(x='epoch',y=['train_mse','valid_mse'])
    plt.title(area+' · datos sintéticos'); plt.ylabel('MSE normalizado'); plt.show()
MET=pd.DataFrame(metricas); display(MET); MET.to_csv(OUT/'metricas_sinteticas.csv',index=False)

**Interpretación:** MAE está en unidades de la variable objetivo; RMSE también. No comparar el MAE de milisegundos con el MAE de cobertura. Con estas simulaciones sencillas es razonable que Ridge o último valor igualen o superen a LTC. Eso enseña cuándo la complejidad no aporta una mejora.

## 13. Sustituir los datos sintéticos por datos propios
Formato recomendado: una fila por entidad y fecha, columnas `entidad`, `tiempo`, características y `objetivo`. No incluir como entrada una variable que revela el objetivo futuro. Separar entrenamiento, validación y prueba por entidad antes de crear secuencias. Las fechas deben conservar orden real; una serie espacial requiere otro nombre y justificación.

La función siguiente recibe datos **ya muestreados a intervalos iguales**. Comprueba esa condición, rechaza faltantes y evita ventanas que mezclen entidades. Predice el objetivo de la observación siguiente, usando ocho observaciones previas. No incluye el objetivo siguiente en las entradas. Si los datos son irregulares, detenerse y diseñar el remuestreo antes de usarla.

In [ ]:
def convertir_tabla(df,features,objetivo,largo=8):
    requeridas=['entidad','tiempo',*features,objetivo]
    faltan=set(requeridas)-set(df.columns)
    if faltan: raise ValueError(f'Faltan columnas: {faltan}')
    if df[requeridas].isna().any().any(): raise ValueError('Definir tratamiento de faltantes primero')
    X=[]; y=[]; grupos=[]; intervalos=[]
    for entidad,g in df.groupby('entidad'):
        g=g.sort_values('tiempo'); tiempos=g.tiempo.to_numpy(dtype=float)
        delta=np.diff(tiempos)
        if len(delta):
            if (delta<=0).any() or not np.allclose(delta,delta[0]):
                raise ValueError(f'Tiempos no uniformes o repetidos en {entidad}')
            intervalos.append(delta[0])
        valores=g[features].to_numpy(np.float32); objetivos=g[objetivo].to_numpy(np.float32)
        for inicio in range(len(g)-largo):
            X.append(valores[inicio:inicio+largo]); y.append(objetivos[inicio+largo]); grupos.append(entidad)
    if intervalos and not np.allclose(intervalos,intervalos[0]):
        raise ValueError('Intervalos diferentes entre entidades: definir escala temporal común')
    if not X: raise ValueError('No hay suficientes registros por entidad')
    return np.stack(X),np.asarray(y),np.asarray(grupos)
# Ejemplo de esquema, aún con datos artificiales.
tabla=pd.DataFrame([{'entidad':p,'tiempo':t,'latencia':600+10*t+p,'dificultad':t%3,'objetivo':600+10*t+p}
                    for p in range(6) for t in range(12)])
X,y,grupos=convertir_tabla(tabla,['latencia','dificultad'],'objetivo')
from sklearn.model_selection import GroupShuffleSplit
train_idx,rest_idx=next(GroupShuffleSplit(n_splits=1,test_size=.4,random_state=11).split(X,y,grupos))
valid_rel,test_rel=next(GroupShuffleSplit(n_splits=1,test_size=.5,random_state=22).split(X[rest_idx],y[rest_idx],grupos[rest_idx]))
valid_idx,test_idx=rest_idx[valid_rel],rest_idx[test_rel]
assert not(set(grupos[train_idx])&set(grupos[test_idx]))
print('X:',X.shape,'entidades entrenamiento:',set(grupos[train_idx]),'prueba:',set(grupos[test_idx]))

## 14. Consideraciones concretas por área
### Pruebas en psicología
Definir primero el constructo y el resultado: latencia, acierto, aprendizaje, fatiga, etc. Una red predictiva no reemplaza evidencia de fiabilidad, validez, normas, equivalencia cultural o invariancia de medición. Separar participantes y revisar práctica, dificultad, dispositivo, idioma y abandono. Si el objetivo es una categoría clínica, evaluar sensibilidad, especificidad, calibración y desempeño entre grupos, no solo accuracy.

### Medición y estadística
Usar series repetidas de instrumentos o sujetos. Distinguir ruido, sesgo, deriva y cambio real. Una predicción precisa de valores observados no identifica automáticamente el valor verdadero. Comparar con regresión y filtros; preservar unidades. Las bandas de incertidumbre requieren un método explícito, como calibración por grupos o modelos probabilísticos; la salida de LTC no es por sí misma un intervalo.

### Cáncer en imágenes
Para seguimiento de lesiones, extraer volumen/área/textura por visita, con escala física y correspondencia anatómica. Un área 2D no es volumen 3D. Separar por paciente; revisar escáner, protocolo, tratamientos y anotaciones. Una clasificación de malignidad requiere etiquetas clínicas adecuadas y validación externa. [TCIA](https://www.cancerimagingarchive.net/browse-collections/) ofrece colecciones; [NLST-New-lesion-LongCT](https://www.cancerimagingarchive.net/analysis-result/nlst-new-lesion-longct/) es una referencia longitudinal para investigar, no un dataset ya preparado en este cuaderno.

### Enfermedades neurológicas en imágenes
Plantear seguimiento longitudinal por paciente o una secuencia espacial declarada. Revisar registro de MRI, modalidad, cambios de escáner y edad. Diagnóstico y evolución son objetivos diferentes. [ADNI](https://adni.loni.usc.edu/data-samples/adni-data/) proporciona datos longitudinales multimodales sujetos a su procedimiento de acceso. No se descargan automáticamente aquí. Comparar LTC con modelos clínicos y modelos de imagen apropiados.

### Plantas y arbustos con drones
Identificar parcela, fechas, GSD, bandas, alturas y anotaciones. Registrar vuelos y controlar iluminación, estación, sensor y altura. Para contar arbustos usar detección/segmentación de instancias y correspondencia entre vuelos; una máscara verde solo mide cobertura. Separar parcelas y campañas para evaluar transferencia. Los [Plant Phenotyping Datasets](https://www.plant-phenotyping.org/datasets-overview) son recursos de fenotipado; no asumimos que todos sean longitudinales ni de drones. Para arbustos propios, diseñar un conjunto de vuelos repetidos con mediciones de campo.

En estas cinco áreas describimos **posibles diseños de investigación**; el artículo LTC no demuestra estos usos.

## 15. Informe de clase: guía para interpretar sin sobreafirmar
Completar después de las corridas:
1. ¿Qué pregunta y qué objetivo operacional estamos evaluando?
2. ¿Cuál es la unidad independiente y cuál es el orden de las mediciones?
3. ¿Qué código, datos, versiones, longitud de secuencia y semillas usamos?
4. ¿Qué corridas fueron cortas y cuáles completas?
5. ¿Qué métrica medimos y cuánto varió entre semillas?
6. ¿Qué modelos de referencia superaron o igualaron a LTC?
7. ¿Qué diferencias existen entre los scripts y el suplemento?
8. ¿Qué fuentes de solapamiento o fuga de información encontramos?
9. ¿Qué se puede proponer con estos datos y qué requiere otro estudio?

Texto orientador: “Bajo esta configuración observamos ____. La diferencia respecto de la publicación fue ____. Proponemos que ____, aunque la interpretación está limitada por ____.”

**Preguntas para discusión:** ¿una red acotada puede equivocarse? ¿por qué HAR no favorece a LTC? ¿cuándo hace falta historia temporal? ¿qué pasa si dos visitas de un paciente caen en entrenamiento y prueba? ¿por qué cobertura vegetal no equivale a conteo de arbustos?

In [ ]:
# Archivar resultados y metadatos. Descargar al terminar para conservarlos fuera de la sesión.
metadatos={'commit':COMMIT,'python_kernel':sys.version,'modo_completo':MODO_COMPLETO,
           'datasets':DATASETS,'modelos':MODELOS,'epocas':EPOCAS,'semillas':SEMILLAS,
           'fecha_utc':pd.Timestamp.now(tz='UTC').isoformat(),
           'alcance':'scripts originales seleccionados + traducción parcial MATLAB + aplicaciones sintéticas'}
(OUT/'metadatos.json').write_text(json.dumps(metadatos,indent=2))
archivo=shutil.make_archive(str(BASE/'resultados_LTC'),'zip',OUT)
print('Resultados:',archivo)
try:
    from google.colab import files
    files.download(archivo)
except ImportError:
    print('Fuera de Colab: abrir el ZIP en la carpeta de trabajo.')

## Anexo: lectura de los resultados piloto de adaptación
Los ejemplos sintéticos son deliberadamente sencillos: un modelo lineal puede representar muy bien su tendencia. Si Ridge gana, eso no contradice el artículo; muestra que la necesidad de LTC depende de la tarea. Probar modelos de referencia es parte del trabajo científico.

Las imágenes de lesión y vegetación de este cuaderno son discos artificiales. Sus métricas no son tasas de detección de cáncer o de plantas reales. Para validar detección serían necesarias anotaciones independientes, datos reales y métricas de localización/segmentación, además de una evaluación temporal si hay seguimiento.

## Verificación incluida en esta entrega
Se validó la estructura del cuaderno y la sintaxis de todas sus celdas. Se ejecutaron las simulaciones, la descarga de ocupación y HAR, TensorFlow 1.14 con Python 3.7, tres corridas originales de ocupación y cinco adaptaciones sintéticas de 40 épocas.

Las pruebas se hicieron en un entorno Linux de preparación, **no en una sesión interactiva del servicio Colab**. El cuaderno usa rutas específicas de Colab cuando está allí y rutas locales fuera de Colab. La instalación sigue dependiendo del acceso a los servidores de paquetes.

**Piloto original de ocupación: 3 épocas, una semilla, 32 unidades. No es la réplica de 200 épocas y cinco semillas.** Los valores corresponden al checkpoint seleccionado por validación; no necesariamente a la última época.

| Modelo | Épocas | Semilla | Accuracy de prueba (%) |
|---|---:|---:|---:|
| ctrnn | 3 | 11 | 97.56 |
| lstm | 3 | 11 | 97.66 |
| ltc | 3 | 11 | 94.00 |

Estas cifras son mediciones realizadas, distintas de las cifras publicadas. No debe interpretarse la diferencia entre modelos de este piloto como evidencia de superioridad general. No se ejecutaron en esta entrega las corridas completas de todos los datasets ni todas las configuraciones del artículo.

En las cinco simulaciones de aplicación se comparó LTC con Ridge y último valor. En varias de ellas Ridge tuvo menor error: esto es un resultado didáctico válido, no un problema que haya que ocultar.


## 16. Fuentes y atribución
- Hasani, R., Lechner, M., Amini, A., Rus, D., & Grosu, R. (2021). *Liquid time-constant networks*. Proceedings of the AAAI Conference on Artificial Intelligence, 35(9), 7657–7666. [Preprint y suplemento](https://arxiv.org/pdf/2006.04439). Tablas de referencia: Tabla 3 de la versión enlazada.
- [Repositorio oficial LTC](https://github.com/raminmh/liquid_time_constant_networks), commit fijado en el cuaderno. Código original bajo Apache 2.0: conservar LICENSE.md y atribución al redistribuir.
- Reyes-Ortiz, J., Anguita, D., Ghio, A., Oneto, L., & Parra, X. (2013). *Human Activity Recognition Using Smartphones* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C54S4K
- [UCI Occupancy Detection](https://archive.ics.uci.edu/dataset/357/occupancy+detection): fuente de sensores para la primera reproducción.
- [Documentación de migración TensorFlow](https://www.tensorflow.org/guide/migrate): explica por qué migrar APIs requiere validar exactitud; este cuaderno elige un intérprete legado separado.
- [TCIA NLST-New-lesion-LongCT](https://www.cancerimagingarchive.net/analysis-result/nlst-new-lesion-longct/), [ADNI Data](https://adni.loni.usc.edu/data-samples/adni-data/) y [Plant Phenotyping Datasets](https://www.plant-phenotyping.org/datasets-overview): fuentes para diseñar futuras aplicaciones, no evidencia de desempeño LTC en ellas.

**Solución de problemas:** si falla UCI, revisar el ZIP y su estructura, sin cambiar silenciosamente de dataset; si falla TF1.14, revisar Python 3.7 y el log del entorno; si se interrumpe Colab, restaurar la carpeta de resultados antes de repetir. La omisión de corridas terminadas no reanuda una época parcialmente ejecutada.

**Estado de verificación del archivo entregado:** se informa en la respuesta de entrega. Las corridas completas originales deben ejecutarse por el estudiante; los resultados propios aparecen únicamente después de ejecutar las celdas correspondientes.